# 00 Feasibility, Baselines and Pretrained SR (Phases 0 to 2)

**Synthetic RAW 50MP Lab.** This notebook builds a 50 MP master from an approximately 4K AI-generated
image with classical interpolation and inference-only pretrained super-resolution (PRD section 30,
Phases 0 to 2):

- **Phase 0:** runtime setup, input loading and validation, colour-space handling, exact 50 MP target
  resolution, 16-bit TIFF and OpenEXR export, memory benchmark.
- **Phase 1:** Lanczos and bicubic baselines, identical crop comparisons, source-consistency metrics and the
  run report.
- **Phase 2:** tiled pretrained super-resolution (SwinIR by default) in the conservative and balanced modes,
  overlap blending, seam checks, memory fallback, and its comparison with the baselines (PRD experiment E02).
- **Final 50 MP image:** every run ends with one final master, chosen by a documented rule (section 18), saved
  as a 16-bit TIFF plus an EXR and delivered to you: downloaded in Colab and copied to Drive when it is mounted.

All processing code lives in the shared `synthetic_raw_lab` package (`src/` in the repository); this notebook
only sets parameters, calls the package and displays results. Face and night processing and sensor simulation
are later phases and are not run here.

## 1. Introduction and scientific limitations

- The source is a rendered, **display-referred** image. Converting it to linear light gives linear-light
  display-referred values, **not scene-referred radiance**. The EXR master is synthetic linear RGB.
- Interpolation estimates samples between source pixels. It cannot add information beyond the source
  sampling limit, so these baselines are the reference that pretrained SR must beat.
- Pretrained super-resolution **synthesises** detail above the source sampling limit from priors learned on other
  photographs. That detail is inferred, not recovered, and is reported separately from the baselines.
- 16-bit integer and floating-point exports add numerical precision, not dynamic range or recovered detail.
- Outputs are **synthetic camera-grade masters**, never camera RAW files. Every report says so.

## 2. Runtime setup

Set the run parameters. In Google Colab these are form fields.

- **Private repository:** add a Colab secret named `GITHUB_TOKEN` (a read-only token for this repository) so
  the setup cell can clone it, or copy the repository to Google Drive and set `PROJECT_DIR`. The token is
  passed to Git for each request only; it is never written to the clone or printed.
- **Code version:** without `PROJECT_DIR`, the setup cell keeps a clone in `/content/experiment_50MP` and
  checks out `REPO_REF` there on every run. A `PROJECT_DIR` is used as it is (cloned from `REPO_REF` only when
  empty). Each run records the commit it used.
- **Keep masters:** a 50 MP run writes several hundred MB. Set `MOUNT_DRIVE` and point `OUTPUT_ROOT` (and
  optionally `REPORTS_ROOT`) at Drive so results survive a Colab VM reset.
- **Presets (PRD 21.6):** `quick` makes a reduced 2 MP output; `standard` the full 50 MP target; `research`
  adds a repeat run of each baseline (reproducibility check) and the memory benchmark.
- **Super-resolution:** `ENABLE_SUPER_RESOLUTION` runs the `SR_MODES` (PRD 10.2) with the `SR_BACKEND` network.
  Weights (about 135 MB for SwinIR) download once into `WEIGHTS_DIR` (default `<project>/weights`; a Drive folder
  keeps them across sessions) and are checksum-verified. Use a GPU runtime: 50 MP SR takes minutes on a GPU and
  much longer on the CPU. Model choice and licences: `docs/models.md`.
- **Final image:** `FINAL_OUTPUT` chooses which output becomes the run's final 50 MP image. `auto` takes the
  conservative SR mode when the category is `human` or a face of at least 96 source pixels is detected, the
  balanced mode for other scenes, and falls back to the next output that finished cleanly (section 18).
  `DOWNLOAD_FINAL` downloads the final 16-bit TIFF through the browser at the end: **about 235 MB** for a
  standard 50 MP run, so it takes a few minutes. With `MOUNT_DRIVE`, the final TIFF and EXR (about 400 MB
  together) are also copied to `FINAL_DRIVE_DIR`, unless `OUTPUT_ROOT` is on Drive already.

In [ ]:
# @title Run parameters
REPO_URL = "https://github.com/RDC28/experiment_50MP.git"  # @param {type:"string"}
REPO_REF = "main"  # @param {type:"string"}
PROJECT_DIR = ""  # @param {type:"string"}
MOUNT_DRIVE = False  # @param {type:"boolean"}
OUTPUT_ROOT = ""  # @param {type:"string"}
REPORTS_ROOT = ""  # @param {type:"string"}
INPUTS_ROOT = ""  # @param {type:"string"}
CONFIG_FILE = "configs/baseline_default.yaml"  # @param {type:"string"}
INPUT_MODE = "upload"  # @param ["upload", "path", "synthetic"]
INPUT_PATH = ""  # @param {type:"string"}
SYNTHETIC_SIZE = "3840x2160"  # @param {type:"string"}
CATEGORY = "universal"  # @param ["human", "night", "universal"]
RUNTIME_PRESET = "standard"  # @param ["quick", "standard", "research"]
TARGET_MEGAPIXELS = 50.0  # @param {type:"number"}
BASELINES = "lanczos,bicubic"  # @param {type:"string"}
RUN_MEMORY_BENCHMARK = True  # @param {type:"boolean"}
ENABLE_SUPER_RESOLUTION = True  # @param {type:"boolean"}
SR_MODES = "conservative,balanced"  # @param {type:"string"}
SR_BACKEND = "swinir"  # @param ["swinir", "realesrgan"]
WEIGHTS_DIR = ""  # @param {type:"string"}
FINAL_OUTPUT = "auto"  # @param ["auto", "lanczos", "bicubic", "sr_conservative", "sr_balanced", "sr_aggressive"]
DOWNLOAD_FINAL = True  # @param {type:"boolean"}
FINAL_DRIVE_DIR = "/content/drive/MyDrive/experiment_50MP/final"  # @param {type:"string"}

## 3. Dependency installation

Finds the repository (`PROJECT_DIR`, the checkout this notebook lives in, or the Colab clone of `REPO_REF`)
and installs the `synthetic_raw_lab` package from it with its dependencies (plus the `sr` extra, PyTorch and
spandrel, when super-resolution is enabled). It reinstalls whenever the
imported package does not come from that repository, so re-running this cell loads the current code. The
package only declares minimum versions, so Colab keeps its preinstalled NumPy, SciPy, Pillow and OpenCV; the
exact versions used are recorded in every run manifest. `requirements.txt` pins the versions the test suite
was run with.

In [ ]:
import base64
import importlib
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

NOTEBOOK_VERSION = "00_Feasibility_Baselines/1.2"
COLAB_CLONE = Path("/content/experiment_50MP")
try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if MOUNT_DRIVE and IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")


def find_project() -> Path:
    if PROJECT_DIR:
        return Path(PROJECT_DIR).expanduser().resolve()
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "pipeline.py").exists():
            return candidate.resolve()
    return COLAB_CLONE


def git(*args: str) -> None:
    env = dict(os.environ)
    token = ""
    if IN_COLAB:
        try:
            from google.colab import userdata

            token = userdata.get("GITHUB_TOKEN") or ""
        except Exception:
            token = ""
    if token:
        credential = base64.b64encode(f"x-access-token:{token}".encode()).decode()
        env.update(
            GIT_CONFIG_COUNT="1",
            GIT_CONFIG_KEY_0="http.extraHeader",
            GIT_CONFIG_VALUE_0=f"Authorization: Basic {credential}",
        )
    result = subprocess.run(["git", *args], env=env, capture_output=True, text=True)
    if result.returncode:
        raise RuntimeError(f"git failed (is GITHUB_TOKEN set for this private repository?): {result.stderr}")


PROJECT = find_project()
if not (PROJECT / "pyproject.toml").exists():
    git("clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(PROJECT))
elif PROJECT == COLAB_CLONE:
    git("-C", str(PROJECT), "fetch", "--depth", "1", REPO_URL, REPO_REF)
    git("-C", str(PROJECT), "checkout", "--detach", "FETCH_HEAD")
spec = importlib.util.find_spec("synthetic_raw_lab")
missing_sr = ENABLE_SUPER_RESOLUTION and importlib.util.find_spec("spandrel") is None
if spec is None or Path(spec.origin).resolve().parent != PROJECT / "src" or missing_sr:
    target = f"{PROJECT}[sr]" if ENABLE_SUPER_RESOLUTION else str(PROJECT)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", target], check=True)
    importlib.invalidate_caches()
for name in [m for m in sys.modules if m.partition(".")[0] == "synthetic_raw_lab"]:
    del sys.modules[name]

import synthetic_raw_lab  # noqa: E402
from synthetic_raw_lab import environment  # noqa: E402

print(f"project: {PROJECT} (Git: {environment.git_revision(PROJECT)})")
print(f"synthetic_raw_lab {synthetic_raw_lab.__version__} from {Path(synthetic_raw_lab.__file__).parent}")

## 4. GPU and environment inspection

Loads the experiment configuration (`CONFIG_FILE` plus the form overrides; empty path fields keep the file's
paths) so every later cell uses the same settings as the pipeline; section 7 shows it in full. Then reports
GPU, CUDA driver version, PyTorch version (if installed), system RAM and the free disk where the masters will
be written (PRD 21.2). The baselines run on the CPU; super-resolution uses CUDA when present, then Apple MPS,
then the CPU.

In [ ]:
from IPython.display import Image, Markdown, display

from synthetic_raw_lab import config as lab_config
from synthetic_raw_lab import environment, ingestion, pipeline, report_builder, resolution, test_images

overrides = {
    "input_category": CATEGORY,
    "runtime_preset": RUNTIME_PRESET,
    "target_megapixels": float(TARGET_MEGAPIXELS),
    "baselines": [b.strip() for b in BASELINES.split(",") if b.strip()],
    "enable_super_resolution": ENABLE_SUPER_RESOLUTION,
    "sr_modes": [m.strip() for m in SR_MODES.split(",") if m.strip()],
    "sr_backend": SR_BACKEND,
    "final_output": FINAL_OUTPUT,
}
paths = {
    "output_root": OUTPUT_ROOT,
    "reports_root": REPORTS_ROOT,
    "inputs_root": INPUTS_ROOT,
    "sr_weights_dir": WEIGHTS_DIR,
}
overrides.update({name: value for name, value in paths.items() if value})
config_path = Path(CONFIG_FILE) if Path(CONFIG_FILE).is_absolute() else PROJECT / CONFIG_FILE
CONFIG = lab_config.load_config(config_path, **overrides)
INPUTS_DIR, OUTPUT_DIR, REPORTS_DIR = pipeline.resolve_roots(CONFIG, PROJECT)

runtime = environment.inspect_runtime(OUTPUT_DIR)
gpu = runtime["gpu"]
print(f"Python {runtime['python']} on {runtime['platform']} (Colab: {runtime['in_colab']})")
print(
    f"GPU: {gpu['name']} {gpu['memory_total_gb']} GB, driver CUDA {gpu['driver_cuda_version']}"
    if gpu
    else "GPU: none detected (super-resolution falls back to Apple MPS or the CPU, much slower)"
)
print(f"PyTorch: {runtime['packages'].get('torch', 'not installed')}")
print(
    f"RAM: {runtime['ram_available_gb']} of {runtime['ram_total_gb']} GB available; "
    f"free disk: {runtime['disk_free_gb']} GB; CPUs: {runtime['cpu_count']}"
)

## 5. Input upload

`upload` asks for a file in Colab, `path` uses `INPUT_PATH` (for example a file on Drive), and `synthetic`
generates the deterministic test image (`SYNTHETIC_SIZE`). Supported formats: PNG, JPEG and TIFF; WebP,
AVIF and OpenEXR when the runtime can decode them. The original file is never modified.

In [ ]:
# Uploads and generated inputs are staged outside the read-only input store, which ingest() copies into.
STAGING = OUTPUT_DIR / "_staging"

if INPUT_MODE == "upload":
    if not IN_COLAB:
        raise RuntimeError("upload only works in Colab; use INPUT_MODE='path' or 'synthetic'")
    from google.colab import files

    uploaded = files.upload()
    if len(uploaded) != 1:
        raise RuntimeError(f"upload exactly one image, got {len(uploaded)}")
    name, data = next(iter(uploaded.items()))
    STAGING.mkdir(parents=True, exist_ok=True)
    SOURCE = STAGING / name
    SOURCE.write_bytes(data)
elif INPUT_MODE == "path":
    SOURCE = Path(INPUT_PATH).expanduser()
elif INPUT_MODE == "synthetic":
    width, height = (int(v) for v in SYNTHETIC_SIZE.lower().split("x"))
    SOURCE = test_images.write_test_image(STAGING / f"synthetic_{width}x{height}.png", width, height)
else:
    raise ValueError(f"unknown INPUT_MODE {INPUT_MODE!r}")
print(f"source: {SOURCE} ({SOURCE.stat().st_size:,} bytes)")

## 6. Input validation

Copies the source into the read-only input store, decodes it without channel swapping or tone changes, and
checks dimensions, aspect ratio, bit depth, channels, alpha, ICC profile, EXIF, file size, clipping,
histograms, compression artefacts, corruption and duplicates (PRD 3.4, 7.1).

In [ ]:
source_image = ingestion.ingest(SOURCE, INPUTS_DIR, CONFIG.input_category, source_color_space=CONFIG.source_color_space)
v = source_image.validation
rows = [
    ("format", f"{v['file']['format']}, {v['file']['size_bytes']:,} bytes, sha256 {v['file']['sha256'][:16]}..."),
    ("stored read-only copy", v["file"]["stored_copy"]),
    (
        "dimensions",
        f"{v['dimensions']['width']} x {v['dimensions']['height']} "
        f"({v['dimensions']['megapixels']:.2f} MP, {v['dimensions']['aspect_ratio_reduced']})",
    ),
    ("bit depth", f"{v['samples']['bit_depth']}-bit (effective {v['samples']['effective_bit_depth']}-bit)"),
    ("channels / alpha", f"{v['samples']['channels_in_file']} / {v['alpha']}"),
    ("ICC profile", v["color"]["icc_profile_description"] or "none"),
    ("interpretation", v["color"]["resolved"]["description"] + f" ({v['color']['resolved']['origin']})"),
    ("working signal", v["color"]["working_signal"]),
    ("EXIF", v["metadata"]["exif"] or "none"),
    ("clipping at max", {k: f"{c['at_max_fraction']:.3%}" for k, c in v["clipping"]["channels"].items()}),
    ("8x8 blockiness", f"{v['compression']['blockiness_ratio']:.3f}"),
    ("duplicates", v["duplicates"] or "none"),
]
display(Markdown("\n".join(["| check | result |", "|---|---|"] + [f"| {k} | {r} |" for k, r in rows])))
for warning in source_image.warnings:
    print("WARNING:", warning)

histogram_path = STAGING / "source_histograms.png"
report_builder.histogram_figure(v["histograms"], histogram_path)
display(Image(filename=str(histogram_path)))

## 7. Experiment configuration

Shows every parameter of the configuration loaded in section 4 with its accepted values, effect and
warnings (PRD 22.2). Then computes the exact target resolution (PRD 3.5), the memory estimate (PRD 21.3) and
the storage estimate (PRD 21.5) against the free space where the masters will be written.

In [ ]:
table = ["| parameter | value | default | accepted | effect | warning |", "|---|---|---|---|---|---|"]
for row in lab_config.parameter_table(CONFIG):
    table.append(
        f"| `{row['parameter']}` | `{row['current']}` | `{row['default']}` | {row['accepted']} | "
        f"{row['effect']} | {row['warning']} |"
    )
display(Markdown("\n".join(table)))

target = resolution.compute_target_resolution(
    source_image.width,
    source_image.height,
    CONFIG.effective_target_megapixels,
    mode=CONFIG.resolution_mode,
    pixel_count_tolerance=CONFIG.pixel_count_tolerance,
    dimension_multiple=CONFIG.dimension_multiple,
    preserve_aspect_ratio=CONFIG.preserve_aspect_ratio,
    max_aspect_ratio_error=CONFIG.max_aspect_ratio_error,
)
print(
    f"target: {target.width} x {target.height} = {target.pixels:,} pixels "
    f"(pixel error {target.pixel_count_error:+.4%}, aspect error {target.aspect_ratio_error:+.5%}, "
    f"scale {target.scale_x:.4f} x {target.scale_y:.4f})"
)
for row in environment.estimate_memory(target.width, target.height):
    print(f"  {row['representation']:<28} {row['estimated_mb']:>8.1f} MB")
storage = environment.estimate_storage(
    target.width,
    target.height,
    len(CONFIG.baselines) + (len(CONFIG.sr_modes) if CONFIG.enable_super_resolution else 0),
    tiff=CONFIG.export_tiff16,
    exr=CONFIG.export_exr,
    exr_bytes_per_sample=2 if CONFIG.exr_precision == "half" else 4,
    source_pixels=source_image.width * source_image.height,
    save_intermediates=CONFIG.save_intermediates,
)
output_free_gb = environment.disk_free_gb(OUTPUT_DIR)
print(
    f"storage upper bound: {storage['upper_bound_bytes'] / 1e9:.2f} GB; free at the output folder: {output_free_gb} GB"
)
if storage["upper_bound_bytes"] > output_free_gb * 1e9:
    print("WARNING: free disk may be insufficient; point OUTPUT_ROOT at Drive or use the quick preset")
del source_image  # the pipeline ingests again from the read-only copy

## 8. Scene analysis

Scene classification is PRD Phase 4. The pipeline does find the main subject: faces with OpenCV's YuNet
detector (its 0.2 MB model downloads once into `WEIGHTS_DIR` and is checksum-verified), or, without a face, a
centre of interest from saliency. The subject frames the first comparison crops (section 16) and decides the
automatic final image (section 18). Detection is uncalibrated and never identity evidence.

## 9. Baseline reconstruction, exports and metrics

Runs the pipeline: optional memory benchmark, then for each baseline (one at a time, so only one 50 MP buffer
is resident) linear-light interpolation, the OpenEXR and 16-bit TIFF masters with read-back validation,
source-consistency metrics, crops and previews. When super-resolution is enabled the same call then runs each
SR mode (section 10) and measures and exports it exactly like the baselines. Last, it saves the chosen output as
the run's final 50 MP image (section 18). Every stage is logged with its PRD 5.2 classification, duration and
peak memory. If a stage fails, outputs already written are kept and the report names the failing stage.

In [ ]:
try:
    RESULT = pipeline.run(
        SOURCE, CONFIG, PROJECT, notebook_version=NOTEBOOK_VERSION, benchmark_memory=RUN_MEMORY_BENCHMARK
    )
except pipeline.StageError as error:
    print(f"FAILED in stage {error.stage}: {error}")
    if error.manifest_path:
        print(f"partial manifest and report: {error.manifest_path.parent}")
    print(
        "Reduced-memory retry: set RUNTIME_PRESET='quick' (or BASELINES or SR_MODES to a single entry) in the "
        "section 2 parameters cell, or a smaller sr_tile_size in the config file, then re-run section 2 and every "
        "section after it. Out-of-memory SR tiles are already split and retried automatically."
    )
    raise
MANIFEST = RESULT.manifest
print(
    f"run {RESULT.run_id}: {MANIFEST['status']} in {MANIFEST['total_seconds']:.1f} s, "
    f"peak RSS {MANIFEST['memory']['pipeline_peak_rss_mb']:.0f} MB"
)

## 10. Pretrained super-resolution

Inference only (PRD 10). Each mode runs the network at its native x4 on overlapping tiles, one strip of tiles
at a time, crossfades the overlaps (the quarter of each overlap next to a tile edge gets no weight), resamples
to the exact target size with Lanczos-3 in linear light, and, unless `preserve_source_structure` is off,
corrects low-frequency drift against the source (PRD 10.3 to 10.5).

- **Mode A, conservative:** the fidelity network (SwinIR classical, pixel loss). Least invented texture.
- **Mode B, balanced:** the perceptual network (SwinIR real-world GAN) with the low-frequency correction.
- **Mode C, aggressive** (experimental, never default): the perceptual network without correction.

Lanczos-3 rings on the crisp edges a network draws (the first balanced Colab master went down to -0.17
linear, clipped to black in the TIFF as dark halos), so every resampled sample is held within the range of the
network pixels under the kernel's main lobe (anti-ringing), and the low-frequency correction works as a gain,
which cannot push a pixel below zero. The `linear min` column shows the lowest value after the resample, after
anti-ringing and after the correction.

The table shows the device, tiling, memory fallbacks and the two seam gates: blend error (how far the
crossfade can move the image from the better-placed tile) and the hard-seam height across each tile boundary
(0 means none), both in 8-bit code values. The heatmaps show where each mode departs from Lanczos interpolation, which is where detail was
synthesised.

In [ ]:
sr_record = MANIFEST.get("super_resolution")
if not sr_record:
    print("super-resolution was not enabled for this run (ENABLE_SUPER_RESOLUTION)")
elif sr_record.get("skipped"):
    print(f"super-resolution skipped: {sr_record['skipped']}")
else:
    rows = [
        "| mode | weights | device | tile / overlap | tiles | inference s | fallbacks | blend error p99 | "
        "hard seam | "
        "vs Lanczos mean / p99 | linear min |",
        "|---|---|---|---|---|---|---|---|---|---|---|",
    ]
    for s in sr_record["modes"].values():
        tiling, seams, model = s["tiling"], s["seams"], s["model"]
        disagreement = tiling["overlap_disagreement"]["blend_error_p99_abs_max"]
        difference = s["interpolation_difference"]
        ranges = report_builder.range_text(s)
        rows.append(
            f"| {s['preset']['label']} | `{s['weights']}` | {model['device']['name']} {model['precision']} | "
            f"{tiling['tile_size']} / {tiling['tile_overlap']} | {tiling['tiles']} | "
            f"{tiling['inference_seconds']:.0f} | {len(tiling['fallbacks'])} | {disagreement} | "
            f"{seams['max_score']} ({'pass' if seams['passed'] else 'FAIL'}) | "
            f"{difference['mean']} / {difference['p99']} | {ranges} |"
        )
    display(Markdown("\n".join(rows)))
    for mode in sr_record["modes"]:
        heatmap = RESULT.reports_dir / "figures" / f"sr_difference_{mode}.png"
        if heatmap.exists():
            display(Image(filename=str(heatmap)))

## 11 to 13. Specialised reconstruction, tonal and optical processing, sensor simulation

Not part of Phases 0 to 2 (PRD Phases 3 and 5).

## 14. Professional exports

16-bit sRGB TIFF (embedded sRGB ICC profile, lossless Deflate, provenance in ImageDescription) and OpenEXR
(R, G, B half or float, linear Rec.709, chromaticities and scene-referred caveat in the header). Each file was
re-read and checked after writing (PRD 17.1, 23.6). SR masters carry their own notice that their fine detail is
synthesised, plus the model weights' checksum. The final image (section 18) is the chosen output's files saved
again under `final/`, verified byte for byte.

In [ ]:
rows = ["| file | kind | size | checks |", "|---|---|---|---|"]
exports = list(MANIFEST.get("intermediates", []))
SR_MODES_RUN = (MANIFEST.get("super_resolution") or {}).get("modes", {})
for summary in [*MANIFEST["baselines"].values(), *SR_MODES_RUN.values()]:
    exports += list(summary["exports"].values())
for e in exports:
    passed = sum(e["checks"].values())
    rows.append(
        f"| `{Path(e['path']).name}` | {e['kind']} | {e['size_bytes'] / 1e6:.1f} MB | {passed}/{len(e['checks'])} |"
    )
for f in (MANIFEST.get("final_output") or {}).get("files", {}).values():
    rows.append(
        f"| `{Path(f['path']).name}` | final image | {f['size_bytes'] / 1e6:.1f} MB | same bytes as `{f['from']}` |"
    )
display(Markdown("\n".join(rows)))
print(f"masters: {RESULT.masters_dir}")

## 15. Quality metrics

Source consistency (PRD 23.2): each 16-bit master is reduced back to the source size by area averaging and
compared with the source. High consistency shows the reconstruction did not drift from the source; it cannot
show that detail above the source resolution is correct. SR modes with the low-frequency correction are pulled
towards the source under the same reduction, so their PSNR and delta E are partly true by construction; edge
displacement stays an independent check. LPIPS needs pretrained weights and is not computed here.

In [ ]:
METHODS = {**MANIFEST["baselines"], **{f"sr_{m}": s for m, s in SR_MODES_RUN.items()}}
names = list(METHODS)
metrics = {m: METHODS[m]["source_consistency"] for m in names}
stats = {m: METHODS[m]["statistics"] for m in names}
rows = [
    ("PSNR (dB)", lambda m: f"{metrics[m]['psnr_db']:.2f}"),
    ("SSIM", lambda m: f"{metrics[m]['ssim']:.5f}"),
    ("MS-SSIM (luma)", lambda m: f"{metrics[m]['ms_ssim']['value']:.5f}"),
    ("edge displacement mean px", lambda m: f"{metrics[m]['edge_displacement']['mean_px']:.3f}"),
    ("delta E 2000 mean", lambda m: f"{metrics[m]['color_difference_ciede2000']['mean']:.4f}"),
    ("L* mean shift", lambda m: f"{metrics[m]['luminance_distribution']['mean_shift']:.4f}"),
    ("samples outside [0, 1]", lambda m: f"{stats[m]['below_zero_fraction'] + stats[m]['above_one_fraction']:.4%}"),
]
table = ["| metric | " + " | ".join(names) + " |", "|---" * (len(names) + 1) + "|"]
table += [f"| {label} | " + " | ".join(fn(m) for m in names) + " |" for label, fn in rows]
display(Markdown("\n".join(table)))

## 16. Visual comparison

Identical crops from every method at 100% output pixels next to the source (nearest-neighbour) and an
amplified difference: the first SR mode against Lanczos when SR ran, otherwise the two kernels. The first crops
show the main subject: the primary face (its eyes, then nose and mouth when the face is larger than a crop),
else the centre of interest; the rest are the most detailed regions, spread apart. Set `crops` in the config
file to choose them yourself. Review them at 100% and 200% for ringing, halos and banding
(PRD 25.2, 26.1).

In [ ]:
subject = MANIFEST["subject"]
print(f"main subject: {report_builder.subject_text(subject)}")
labels = {c["name"]: c["label"] for c in MANIFEST["crops"]}
for panel in MANIFEST.get("crop_panels", []):
    print(f"crop {panel['crop']}: {labels.get(panel['crop'], '')}")
    display(Image(filename=panel["path"]))

## 17. Report generation

The Markdown report and machine-readable manifest are written next to their figures (PRD 27). The conclusions
rank every output (baselines and SR modes) on each metric and state which output is the final image and why.

In [ ]:
print(f"report:   {RESULT.report_path}")
print(f"manifest: {RESULT.manifest_path}")
conclusions = report_builder.conclusions(MANIFEST)
display(Markdown("\n".join(f"- {line}" for line in conclusions)))

## 18. Final 50 MP image, download and archive

**The final image.** One output becomes the run's final 50 MP image (PRD 27.3, 32.3). With
`FINAL_OUTPUT = "auto"`: the conservative SR mode when the category is `human` or the primary face is at least
96 source pixels (PRD 10.2 prefers it for faces, PRD 11.2 tier 1), otherwise the balanced mode (the default for
general scenes). An output that failed export validation or a seam gate is passed over for the next one, down
to Lanczos and bicubic; the aggressive mode is never chosen automatically. A named `FINAL_OUTPUT` picks that
output. Its 16-bit TIFF and EXR are saved under `final/` in the masters folder with `final_` in front of their
names.

**Delivery.** In Colab, `DOWNLOAD_FINAL` downloads the final 16-bit TIFF: **about 235 MB** for a standard 50 MP
run, so the browser download takes a few minutes (if it stalls, take the file from Drive or the Files panel).
With Drive mounted, the final TIFF, EXR and their provenance sidecars (about 400 MB) are copied to
`FINAL_DRIVE_DIR/<run id>`, unless the masters are on Drive already. The other masters (about 1.7 GB for four
outputs) stay in `OUTPUT_ROOT`.

**Report archive.** The report and figures are zipped and downloaded separately (PRD 21.5); the zip holds no
masters.

In [ ]:
from synthetic_raw_lab import final_output

FINAL = MANIFEST["final_output"]
if FINAL.get("method"):
    rule = "requested" if FINAL["rule"] == "override" else "automatic rule"
    display(Markdown(f"**Final 50 MP image: {FINAL['method']}** ({rule}): {FINAL['reason']}."))
    for note in FINAL.get("notes", []):
        print("NOTE:", note)
    preview = RESULT.reports_dir / "figures" / "final_preview.jpg"
    if preview.exists():
        display(Image(filename=str(preview)))
    for f in FINAL["files"].values():
        print(f"{Path(f['path']).name}: {f['width']} x {f['height']}, {f['size_bytes'] / 1e6:,.0f} MB ({f['path']})")
else:
    print(f"no final image: {FINAL.get('reason')}")
DELIVERY = final_output.deliver(MANIFEST, download=DOWNLOAD_FINAL, drive_dir=FINAL_DRIVE_DIR if MOUNT_DRIVE else None)

In [ ]:
import shutil

archive = shutil.make_archive(str(RESULT.reports_dir), "zip", root_dir=RESULT.reports_dir)
print(f"report archive (report and figures, no masters): {archive}")
if IN_COLAB:
    from google.colab import files

    files.download(archive)